# Hesak – Step 17: TEST ROUND 5 on `data/test_round5` (run ONCE)

**What is tested:** the frozen system **v11 final** (team decision, RS 2026-11-11 00:38, log section 8):
- labels **v1.9** (internal) + head **`hesak_head_v11.keras`** + **`hesak_head_v11_config.json`** (no retraining, no threshold change),
- **v11 output layer (16 sounds)** = v2.0 without phone_tone: merges, removed sounds, and the priority rule (alarm hidden when emergency_siren fires),
- **alarm = fire alarm / smoke detector only**; **rain / storm** = rain alone or rain with thunder; **birds** = YAMNet alone at 0.475.

**Compared with (same clips, same v11 output layer):** YAMNet mapping alone (labels v1.7, the baseline of every round), v6b (rounds 2–3) and v10 (round 4).

**Why these numbers are honest:** the clips in `data/test_round5` were collected by RS AFTER v11 was frozen. YouTube links are checked by Claude
against all notes files before upload, and Cell 5 removes any clip that is a Freesound ID used anywhere, a file name used anywhere else in our data,
or an exact copy (same bytes) of a training / earlier-round clip (rounds 2–4, `train_extra`, `alarm_excluded`). Nothing is changed after this run.
**Phone tones in `other`:** phone_tone is no longer a supported sound, so phone ringtones in `other` count as false alarms if any alert fires (RS 11 Nov).

**Performance targets (NFR):** written in Cell 2 **before** collecting (approved by RS, 2026-11-11 00:38, same targets as rounds 2–4).

**How to run:** Runtime -> Run all, **ONCE**. Cell 3 stops the notebook if round 5 was already run. Wait until Cell 9 prints `Saved: ...`, then send the downloaded `.ipynb`.
**Outputs (`outputs/`):** `step17_round5_per_file.csv`, `step17_round5_per_sound.csv`, `step17_round5_summary.json`.

In [1]:
# ---------------------------------------------------------------
# Cell 1: Library versions (the report's technology table needs them)
# ---------------------------------------------------------------
import sys, platform, importlib
print('Python', sys.version.split()[0], '|', platform.platform())
for name in ['numpy', 'pandas', 'librosa', 'soundfile', 'tensorflow', 'tensorflow_hub', 'keras']:
    try:
        module = importlib.import_module(name)          # import by name so a missing library does not crash the cell
        print(f'{name:16s}', getattr(module, '__version__', '?'))
    except Exception as e:
        print(f'{name:16s} NOT INSTALLED', e)

Python 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
numpy            2.4.6
pandas           2.2.3
librosa          0.11.0
soundfile        0.14.0
tensorflow       2.21.0


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


tensorflow_hub   0.16.1
keras            3.13.2


In [2]:
# ---------------------------------------------------------------
# Cell 2: PERFORMANCE TARGETS (NFR) and tiers, fixed BEFORE seeing any round-5 result
#   detection = share of a sound's clips where its own (v2.0) alert fired
#   false alarms = share of 'other' clips (sounds we do not support) where ANY alert fired
#   Approved by RS on 2026-11-11 00:38 (log section 8), before collecting: the same targets as rounds 2, 3 and 4.
#   Not reaching a target is acceptable if reported honestly; changing a target after the run is not.
# ---------------------------------------------------------------
NFR = {
    'critical_detection':  0.80,   # emergency_siren, alarm
    'important_detection': 0.75,   # knock, bell, scream
    'mean_detection':      0.70,   # all 16 v11 sounds (v2.0 without phone_tone)
    'false_alarm_other':   0.20,   # at most
}
TIERS2 = {'critical': ['emergency_siren', 'alarm'],                 # v2.0 tiers (the critical category is locked in the app)
          'important': ['knock', 'bell', 'scream']}                  # every other v2.0 sound is 'rest'
print('targets:', NFR)
print('tiers:', TIERS2)

targets: {'critical_detection': 0.8, 'important_detection': 0.75, 'mean_detection': 0.7, 'false_alarm_other': 0.2}
tiers: {'critical': ['emergency_siren', 'alarm'], 'important': ['knock', 'bell', 'scream']}


In [3]:
# ---------------------------------------------------------------
# Cell 3: Drive, paths, the frozen system files and the v2.0 output layer (nothing here is tuned)
# ---------------------------------------------------------------
from google.colab import drive
import os, json, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')                                          # librosa prints many harmless format warnings
drive.mount('/content/drive')                                              # choose the team Google account (email removed for the public repo)

BASE = '/content/drive/MyDrive/Colab Notebooks/Hesak_YAMNet'
OUT = f'{BASE}/outputs'
TEST = f'{BASE}/data/test_round5'                                          # the round-5 test folders
CACHE = f'{OUT}/cache_test_round5'                                         # YAMNet outputs of the round-5 clips
# Safety (lesson of 10 Nov, log 6.20 note 7): a test round runs ONCE. If its results already exist, stop here.
assert not os.path.exists(f'{OUT}/step17_round5_summary.json'), \
    'Round 5 has ALREADY been run (step17_round5_summary.json exists). Do NOT run it again.'
os.makedirs(CACHE, exist_ok=True)

L20  = json.load(open(f'{OUT}/hesak_labels.json', encoding='utf-8'))       # v2.0 (17 sounds); v11 hides phone_tone -> 16
L    = json.load(open(f"{OUT}/{L20['internal_labels_file']}", encoding='utf-8'))  # v1.9: the frozen internal labels
L17  = json.load(open(f'{OUT}/hesak_labels_v1.7.json', encoding='utf-8'))  # v1.7: baseline + v4 (round-1 systems)
CFG  = json.load(open(f'{OUT}/hesak_head_v11_config.json'))               # FROZEN system of round 5 (v11 final)
CFG10 = json.load(open(f'{OUT}/hesak_head_v10_config.json'))              # system of round 4, for comparison
CFG6 = json.load(open(f'{OUT}/hesak_head_v6b_config.json'))               # system of rounds 2 and 3, for comparison
assert L20['version'] == '2.0' and L['version'] == '1.9' and CFG['version'] == 'head_v11' and CFG10['version'] == 'head_v10' \
    and CFG6['version'] == 'head_v6b', (L20['version'], L['version'], CFG['version'], CFG10['version'], CFG6['version'])
REMOVED = set(CFG['output_layer']['removed_sounds'])                       # v11 output layer: sounds no longer shown (phone_tone)
assert REMOVED == {'phone_tone'} and abs(CFG['mapping_thresholds']['birds'] - 0.475) < 1e-9   # RS decisions of 10 Nov
ALLS = [s for s, v in L['sounds'].items() if v['release'] == 'R1']        # the 25 internal R1 sounds (v1.9)
R1V2 = [s for s, v in L20['sounds'].items() if v['release'] == 'R1' and s not in REMOVED]   # the 16 sounds shown to the user (v11)
TIERS = CFG['tiers']                                                        # internal tiers (used inside the v6b system)
TIER = {s: t for t, v in TIERS2.items() for s in v}                         # v2.0 tier of each sound ('rest' if missing)

TO_V2 = {i: s for s, v in L20['sounds'].items() for i in v['internal_sounds'] if s not in REMOVED}   # internal -> shown name
PRIO2 = {k: v for k, v in L20['priority'].items() if not k.startswith('_')}       # v2.0 priority rule
def to_v2(internal_alerts):
    """v11 output layer: v2.0 merge, drop removed sounds (now also phone_tone), then the v2.0 priority rule."""
    out = {TO_V2[a] for a in internal_alerts if a in TO_V2}
    return {s for s in out if not (s in PRIO2 and any(p in out for p in PRIO2[s]))}

assert set(TIERS2['critical']) | set(TIERS2['important']) <= set(R1V2) and len(R1V2) == 16
print('labels', L20['version'], '(internal', L['version'] + ') | config', CFG['version'], '| model', CFG['model_file'])
print('internal sounds:', len(ALLS), '| v11 sounds shown:', len(R1V2), R1V2, '| removed:', sorted(L20['removed_internal']))

Mounted at /content/drive
labels 2.0 (internal 1.9) | config head_v11 | model hesak_head_v11.keras
internal sounds: 25 | v11 sounds shown: 16 ['emergency_siren', 'alarm', 'bell', 'knock', 'keys', 'scream', 'baby_cry', 'crowd', 'washing_machine', 'horn', 'dog', 'cat', 'birds', 'rooster', 'livestock', 'rain'] | removed: ['clapping', 'explosion', 'footsteps', 'skid', 'traffic']


In [4]:
# ---------------------------------------------------------------
# Cell 4: YAMNet, the mapping rules, both heads, and the three systems (same code as steps 9 / 14)
# ---------------------------------------------------------------
import tensorflow as tf, tensorflow_hub as hub, keras, librosa
from collections import Counter
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
class_names = pd.read_csv(yamnet.class_map_path().numpy().decode())['display_name'].tolist()
idx = {n: i for i, n in enumerate(class_names)}
WIN_F = 3                                                                  # 3 YAMNet frames per head window

def run_score(x, n):
    """Best score over any n consecutive steps (each run is scored by its weakest step)."""
    if len(x) < n:
        return float(x.min())
    return float(np.stack([x[i:len(x) - n + 1 + i] for i in range(n)]).min(axis=0).max())

class Mapping:
    """YAMNet classes -> our sounds for one labels file (suppress rules, min_frames, priority)."""
    def __init__(self, labels):
        S = labels['sounds']
        self.sounds = [s for s in S if S[s]['yamnet'] and S[s]['release'] == 'R1']
        self.cls = [[idx[c] for c in S[s]['yamnet']] for s in self.sounds]
        self.sup = {self.sounds.index(s): [idx[c] for c in v] for s, v in labels['suppress'].items() if s in self.sounds}
        self.thr = {s: float(labels['rules']['thresholds'][s]) for s in self.sounds}
        self.nmin = {s: int(labels['rules']['min_frames'][s]) for s in self.sounds}
        self.priority = {k: v for k, v in labels.get('priority', {}).items() if not k.startswith('_')}
    def scores(self, raw, mask):
        Sc = np.stack([raw[:, c].max(axis=1) for c in self.cls], axis=1)
        for k, sup in self.sup.items():
            Sc[raw[:, sup].max(axis=1) >= Sc[:, k], k] = 0
        Sc[~mask] = 0
        return {s: run_score(Sc[:, k], self.nmin[s]) for k, s in enumerate(self.sounds)}
    def show(self, fired):
        """Priority display rule: the more specific alert hides the general one."""
        return {s for s in fired if not (s in self.priority and any(p in fired for p in self.priority[s]))}

M19, M17 = Mapping(L), Mapping(L17)
head11 = keras.models.load_model(f"{OUT}/{CFG['model_file']}", compile=False)    # v11 head (16 sounds), FROZEN
head10 = keras.models.load_model(f"{OUT}/{CFG10['model_file']}", compile=False)  # v10 head (round 4)
head6 = keras.models.load_model(f"{OUT}/{CFG6['model_file']}", compile=False)    # v6 head (15 sounds)
assert head11.output_shape[-1] == len(CFG['classes']) and head6.output_shape[-1] == len(CFG6['classes'])

def head_scores(emb, act, net):
    """Per head sound: (2-window score, 1-window score). 2-window = best probability that 2 consecutive 3-frame
    windows both reach; 1-window = best single window (mode 'head_short'). Mostly-silent windows = 0."""
    n = len(emb)
    if n < WIN_F:
        P = net(emb.mean(0, keepdims=True), training=False).numpy()
    else:
        a = act if act.any() else np.ones(n, bool)
        P = net(np.stack([emb[i:i + WIN_F].mean(0) for i in range(n - WIN_F + 1)]), training=False).numpy()
        ok = np.array([a[i:i + WIN_F].sum() >= 2 for i in range(n - WIN_F + 1)])
        if ok.any():
            P[~ok] = 0
    return (np.array([run_score(P[:, k], 2) for k in range(P.shape[1])]),
            np.array([run_score(P[:, k], 1) for k in range(P.shape[1])]))

def fire(mode, h, mp):
    return {'head': h, 'mapping': mp, 'head_or_mapping': h or mp, 'head_and_mapping': h and mp}[mode]

def system_from_config(cfg, net):
    """One frozen system: every internal sound by its config option, mapping at the config thresholds, then priority (v1.9)."""
    def run(raw, emb, act):
        mask = act if act.any() else np.ones(len(raw), bool)
        ms = M19.scores(raw, mask)
        h2, h1 = head_scores(emb, act, net)
        out = set()
        for s in ALLS:
            t = cfg['mapping_thresholds'].get(s)                           # None when the sound uses the head alone
            mp = s in ms and t is not None and ms[s] >= t
            if s in cfg['classes']:
                k, mode = cfg['classes'].index(s), cfg['mode'][s]
                score = h1[k] if mode == 'head_short' else h2[k]           # head_short: ONE window above the threshold is enough
                h = score >= cfg['thresholds'][s]
                if fire('head' if mode == 'head_short' else mode, h, mp):
                    out.add(s)
            elif mp:
                out.add(s)
        return M19.show(out)
    return run

system_v11 = system_from_config(CFG, head11)                               # FROZEN round-5 system
system_v10 = system_from_config(CFG10, head10)                             # round-4 system (its own thresholds)
system_v6b = system_from_config(CFG6, head6)                               # rounds 2 and 3 system

def system_mapping(raw, emb, act):
    """Baseline: YAMNet mapping alone with labels v1.7 (same baseline as round 1)."""
    mask = act if act.any() else np.ones(len(raw), bool)
    return M17.show({s for s, v in M17.scores(raw, mask).items() if v >= M17.thr[s]})
print('systems ready: mapping (v1.7) | v6b (rounds 2-3) | v10 (round 4) | v11 (FROZEN, round 5) | v11 modes:', dict(Counter(CFG['mode'].values())))

systems ready: mapping (v1.7) | v6b (rounds 2-3) | v10 (round 4) | v11 (FROZEN, round 5) | v11 modes: {'mapping': 9, 'head_and_mapping': 1, 'head': 14, 'head_or_mapping': 1}


In [5]:
# ---------------------------------------------------------------
# Cell 5: The round-5 clips, after the leakage checks
#   removed: Freesound IDs used in training (FSD50K, all versions), IDs anywhere else in our data,
#            duplicates, folders that are not a v2.0 sound,
#            a FILE NAME used anywhere else in data/, or an EXACT COPY (same bytes, md5) of a clip in
#            train_extra, test_round2, test_round3, test_round4 or alarm_excluded (all training recordings included)
#   (YouTube links were checked by hand against all notes files before upload; see the _notes_<sound>.txt files)
# ---------------------------------------------------------------
AUDIO_EXT = {'.wav', '.mp3', '.flac', '.aiff', '.aif', '.ogg', '.m4a', '.mp4', '.opus', '.webm'}
used_fsd = set(pd.read_csv(f'{OUT}/step5_fsd50k_clips_used.csv', dtype={'fname': str}).fname)
import hashlib
def md5(path):
    with open(path, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()
ours, names, hashes = set(), set(), set()
for root, _, files in os.walk(f'{BASE}/data'):
    if root.startswith(TEST):
        continue
    ours |= {f.split('__')[0] for f in files if '__' in f}                  # Freesound IDs anywhere in our data
    audio = [f for f in files if os.path.splitext(f)[1].lower() in AUDIO_EXT]
    names |= set(audio)                                                    # every audio file name in our data
    if any(k in root for k in ['train_extra', 'test_round2', 'test_round3', 'test_round4', 'alarm_excluded']):
        hashes |= {md5(f'{root}/{f}') for f in audio}                      # exact copies of recent recordings
print('Freesound IDs:', len(ours), '| file names:', len(names), '| hashed clips:', len(hashes))
rows, removed, seen = [], [], set()
for sound in sorted(os.listdir(TEST)):
    if not os.path.isdir(f'{TEST}/{sound}'):
        continue
    for f in sorted(os.listdir(f'{TEST}/{sound}')):
        if os.path.splitext(f)[1].lower() not in AUDIO_EXT:                # skips the _notes_*.txt files
            continue
        fid = f.split('__')[0] if '__' in f else None                      # only Freesound names contain '__'
        why = ('used in FSD50K training' if fid in used_fsd else 'already in our data' if fid in ours
               else 'duplicate' if fid in seen else None) if fid else None
        if not why and f in names:
            why = 'same file name elsewhere in our data'
        if not why:
            h = md5(f'{TEST}/{sound}/{f}')
            why = 'exact copy of a training / earlier-round clip' if h in hashes else 'exact copy inside round 5' if h in seen else None
            seen.add(h)
        if sound != 'other' and sound not in R1V2:
            why = 'folder is not a v11 sound'
        if why:
            removed.append((sound, f, why)); continue
        if fid:
            seen.add(fid)
        rows.append(dict(sound=sound, file=f))
T = pd.DataFrame(rows, columns=['sound', 'file'])
print('test clips:', len(T), '| removed:', len(removed))
for r in removed:
    print('  removed:', r)
display(T.sound.value_counts().to_frame('clips').T)
missing = [s for s in R1V2 if s not in set(T.sound)]
print('v11 sounds with no round-5 clips:', missing or 'none')
assert (T.sound == 'other').sum() >= 5, 'test_round5/other needs at least 5 clips to measure false alarms'

Freesound IDs: 489 | file names: 989 | hashed clips: 435
test clips: 96 | removed: 0


sound,other,alarm,baby_cry,birds,bell,crowd,dog,emergency_siren,cat,horn,keys,knock,livestock,rain,rooster,scream,washing_machine
clips,16,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5


v11 sounds with no round-5 clips: none


In [6]:
# ---------------------------------------------------------------
# Cell 6: Run the three systems on every clip, then the v2.0 output layer
#   *_internal = alerts with the internal names (kept for diagnosis), * = alerts the user would see (v2.0)
# ---------------------------------------------------------------
from tqdm.auto import tqdm
SR, MAX_SEC, HOP, WIN, ACTIVE_REL = 16000, 600, 0.48, 0.96, 0.10
safe = lambda s: re.sub(r'[^A-Za-z0-9_.-]', '_', s)

def yamnet_cached(path, cache_path):
    """YAMNet outputs + active frames for one clip, computed once and stored on Drive."""
    if os.path.exists(cache_path):
        z = np.load(cache_path); return z['raw'].astype(np.float32), z['emb'].astype(np.float32), z['active']
    y, _ = librosa.load(path, sr=SR, mono=True, duration=MAX_SEC)
    y = np.pad(y, (0, max(0, 15600 - len(y)))).astype(np.float32)          # YAMNet needs at least 0.975 s
    raw, emb, _ = yamnet(y); raw, emb = raw.numpy(), emb.numpy()
    w, h = int(WIN * SR), int(HOP * SR)
    rms = np.array([np.sqrt(np.mean(y[i*h:i*h+w] ** 2) + 1e-12) for i in range(len(raw))])
    act = rms >= ACTIVE_REL * rms.max()                                    # active = loudness >= 10% of the loudest frame
    np.savez_compressed(cache_path, raw=raw.astype(np.float16), emb=emb.astype(np.float16), active=act)
    return raw, emb, act

SYSTEMS = {'mapping': system_mapping, 'v6b': system_v6b, 'v10': system_v10, 'v11': system_v11}
res = []
for _, r in tqdm(T.iterrows(), total=len(T)):
    out = yamnet_cached(f'{TEST}/{r.sound}/{r.file}', f'{CACHE}/{safe(r.sound + "__" + r.file)}.npz')
    row = dict(sound=r.sound, file=r.file)
    for k, f in SYSTEMS.items():
        internal = f(*out)                                                 # internal alerts (v1.9 / v1.7 names)
        row[f'{k}_internal'] = sorted(internal)
        row[k] = sorted(to_v2(internal))                                   # what the user sees (v2.0)
    res.append(row)
D = pd.DataFrame(res)
print('done:', len(D))

  0%|          | 0/96 [00:00<?, ?it/s]

done: 96


In [7]:
# ---------------------------------------------------------------
# Cell 7: RESULTS per sound, per tier, and false alarms (v2.0 names)
#   detected     = the sound's own v2.0 alert fired
#   wrong_alerts = another v2.0 alert fired too (allowed extras not counted)
#   critical sounds: also 'any critical alert' (the user is warned even if the name is different)
# ---------------------------------------------------------------
from collections import Counter
ALLOWED_EXTRA = {'rooster': {'birds'}}                                     # a rooster is also a bird
CRIT = set(TIERS2['critical'])
rows = []
for sound, g in D[D.sound != 'other'].groupby('sound'):
    extra = ALLOWED_EXTRA.get(sound, set())
    r = dict(tier=TIER.get(sound, 'rest'), sound=sound, clips=len(g))
    for k in SYSTEMS:
        r[f'{k}_detected'] = np.mean([sound in f for f in g[k]])
        r[f'{k}_wrong'] = np.mean([any(x != sound and x not in extra for x in f) for f in g[k]])
    r['v11_any_critical'] = np.mean([bool(set(f) & CRIT) for f in g['v11']]) if sound in CRIT else np.nan
    r['v11_wrong_with'] = ', '.join(f'{s} {n}' for s, n in Counter(x for f in g['v11'] for x in f if x != sound).most_common(3))
    rows.append(r)
RES = pd.DataFrame(rows)
RES['tier'] = pd.Categorical(RES.tier, ['critical', 'important', 'rest'])
RES = RES.sort_values(['tier', 'sound']).reset_index(drop=True)
display(RES.style.format({c: '{:.0%}' for c in RES.columns if c.endswith(('_detected', '_wrong', '_any_critical'))}, na_rep='-'))

O = D[D.sound == 'other']
SUM = {}
for k in SYSTEMS:
    det = lambda tier: float(RES[RES.tier == tier][f'{k}_detected'].mean())
    SUM[k] = dict(false_alarm_other=float(np.mean([len(f) > 0 for f in O[k]])),
                  critical_detection=det('critical'), important_detection=det('important'),
                  rest_detection=det('rest'), mean_detection=float(RES[f'{k}_detected'].mean()))
    print(f"{k:8s} | false alarms on other ({len(O)} clips): {SUM[k]['false_alarm_other']:.0%} | detection: critical {SUM[k]['critical_detection']:.0%}, "
          f"important {SUM[k]['important_detection']:.0%}, rest {SUM[k]['rest_detection']:.0%}, all {SUM[k]['mean_detection']:.0%} |",
          'alerts on other:', Counter(s for f in O[k] for s in f).most_common(4))
print('\nv11 alerts on each other clip:')
for _, r in O.iterrows():
    print(f'  {r.file:40s} ->', ', '.join(r.v11) or '-')
print('\nv11 alerts on each alarm clip (information: which alarm kind is missed):')
for _, r in D[D.sound == 'alarm'].iterrows():
    print(f'  {r.file:40s} ->', ', '.join(r.v11) or '-')

,tier,sound,clips,mapping_detected,mapping_wrong,v6b_detected,v6b_wrong,v10_detected,v10_wrong,v11_detected,v11_wrong,v11_any_critical,v11_wrong_with
0,critical,alarm,5,100%,0%,100%,100%,100%,0%,100%,0%,100%,
1,critical,emergency_siren,5,100%,0%,100%,0%,100%,0%,100%,20%,100%,horn 1
2,important,bell,5,100%,40%,100%,0%,100%,0%,100%,20%,-,horn 1
3,important,knock,5,60%,0%,80%,0%,80%,0%,60%,0%,-,
4,important,scream,5,0%,0%,80%,0%,80%,0%,80%,20%,-,dog 1
5,rest,baby_cry,5,100%,20%,100%,20%,100%,40%,100%,40%,-,"scream 1, crowd 1"
6,rest,birds,5,100%,0%,100%,0%,100%,0%,100%,0%,-,
7,rest,cat,5,100%,20%,100%,20%,100%,20%,100%,20%,-,livestock 1
8,rest,crowd,5,80%,0%,80%,60%,80%,20%,100%,20%,-,scream 1
9,rest,dog,5,100%,0%,100%,0%,100%,0%,100%,0%,-,


mapping  | false alarms on other (16 clips): 19% | detection: critical 100%, important 53%, rest 75%, all 74% | alerts on other: [('bell', 1), ('keys', 1), ('alarm', 1)]
v6b      | false alarms on other (16 clips): 31% | detection: critical 100%, important 87%, rest 78%, all 82% | alerts on other: [('knock', 2), ('keys', 1), ('scream', 1), ('alarm', 1)]
v10      | false alarms on other (16 clips): 25% | detection: critical 100%, important 87%, rest 82%, all 85% | alerts on other: [('knock', 2), ('keys', 1), ('scream', 1), ('bell', 1)]
v11      | false alarms on other (16 clips): 19% | detection: critical 100%, important 80%, rest 89%, all 89% | alerts on other: [('horn', 1), ('keys', 1), ('scream', 1)]

v11 alerts on each other clip:
  other_RS_r5_01.ogg                       -> -
  other_RS_r5_02.ogg                       -> -
  other_RS_r5_03.ogg                       -> -
  other_RS_r5_04.ogg                       -> -
  other_RS_r5_05.ogg                       -> horn
  other_RS_r5

In [8]:
# ---------------------------------------------------------------
# Cell 8: Targets (NFR) check for the FROZEN system (v11 final + v11 output layer, 16 sounds)
# ---------------------------------------------------------------
s6 = SUM['v11']
checks = [('critical_detection', s6['critical_detection'] >= NFR['critical_detection']),
          ('important_detection', s6['important_detection'] >= NFR['important_detection']),
          ('mean_detection', s6['mean_detection'] >= NFR['mean_detection']),
          ('false_alarm_other', s6['false_alarm_other'] <= NFR['false_alarm_other'])]
for name, ok in checks:
    sign = '<=' if name == 'false_alarm_other' else '>='
    print(f"{name:20s} {s6[name]:.0%}  (target {sign} {NFR[name]:.0%})  ->  {'PASS' if ok else 'NOT REACHED'}")
print(f'\n{sum(ok for _, ok in checks)} of {len(checks)} targets reached. Report the result as it is.')

critical_detection   100%  (target >= 80%)  ->  PASS
important_detection  80%  (target >= 75%)  ->  PASS
mean_detection       89%  (target >= 70%)  ->  PASS
false_alarm_other    19%  (target <= 20%)  ->  PASS

4 of 4 targets reached. Report the result as it is.


In [9]:
# ---------------------------------------------------------------
# Cell 9: Save the round-5 results (these are the numbers for the report). The summary file also blocks a second run (Cell 3).
# ---------------------------------------------------------------
LISTS = [c for c in D.columns if c not in ('sound', 'file')]
D.assign(**{c: D[c].map(', '.join) for c in LISTS}).to_csv(f'{OUT}/step17_round5_per_file.csv', index=False, encoding='utf-8-sig')
RES.to_csv(f'{OUT}/step17_round5_per_sound.csv', index=False, encoding='utf-8-sig')
json.dump(dict(labels=L20['version'], internal_labels=L['version'], decision=CFG['version'], model=CFG['model_file'],
               nfr=NFR, tiers=TIERS2, test_clips=len(D), other_clips=int(len(O)), summary=SUM,
               nfr_reached={n: bool(ok) for n, ok in checks}, sounds_without_clips=missing, removed=removed),
          open(f'{OUT}/step17_round5_summary.json', 'w'), indent=2)
print('Saved: step17_round5_per_file.csv, step17_round5_per_sound.csv, step17_round5_summary.json')

Saved: step17_round5_per_file.csv, step17_round5_per_sound.csv, step17_round5_summary.json


## How to read the results
- **Cell 5:** clips per folder, and which clips were removed (leakage checks). Expected: about 95 clips, 0 removed.
- **Cell 7:** per sound, the four systems side by side (v11 names, 16 sounds); one summary line per system; the v11 alerts on every `other` clip and on every alarm clip.
- **Cell 8:** the targets (NFR) for the frozen system v11.
- 5 clips per sound: one clip = 20%. Report the numbers with this caution.
- Run it **ONCE**. Send the downloaded `.ipynb`. **The results discussion in the report is written by the team.**